# ModernBERT Epoch 1 — RouterArena 평가

이 notebook은 Google Drive에 저장된 `step-00003476` checkpoint만 불러와 RouterArena full 8,375행을 평가합니다. 학습, gradient 계산, optimizer update와 epoch 2 진행은 수행하지 않습니다.

1. Colab에서 T4 GPU runtime을 선택합니다.
2. `modernbert-colab-full-input.tar.gz`를 `/content`에 업로드합니다.
3. 아래 셀을 순서대로 실행합니다.
4. T4 OOM을 피하기 위해 RouterArena batch size는 1로 고정합니다.

완료 결과는 `MyDrive/LLM_Router/training/modernbert-router-baseline-v1/routerarena-epoch-evaluations/epoch-01/`에 공식 epoch 평가 artifact 형식으로 저장됩니다. 이후 전체 학습 notebook을 재개하면 완료된 epoch 1 평가는 다시 실행하지 않습니다.

In [ ]:
import platform
import subprocess
from pathlib import Path

from google.colab import drive

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print({"python": platform.python_version(), "gpu": gpu.stdout.strip() or None})
if gpu.returncode != 0:
    raise RuntimeError("GPU runtime이 아닙니다. Colab에서 T4 GPU runtime을 선택하세요.")

drive.mount("/content/drive", force_remount=False)
DRIVE_RUN = Path(
    "/content/drive/MyDrive/LLM_Router/training/modernbert-router-baseline-v1"
)
print("Persistent run:", DRIVE_RUN)

In [ ]:
import shutil
import tarfile

archive = Path("/content/modernbert-colab-full-input.tar.gz")
workspace = Path("/content/LLM_Router")
if not archive.is_file():
    raise FileNotFoundError(
        "modernbert-colab-full-input.tar.gz를 Colab /content에 먼저 업로드하세요."
    )
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
with tarfile.open(archive, "r:gz") as bundle:
    bundle.extractall(workspace, filter="data")
print("Extracted:", workspace)

In [ ]:
import importlib.util
import os
import subprocess
import sys

def install(label, *arguments):
    print(f"\n[install] {label}", flush=True)
    command = [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-cache-dir",
        "--break-system-packages", *arguments,
    ]
    result = subprocess.run(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} installation failed with exit code {result.returncode}"
        )
    return result.stdout

PYTORCH_INDEX = "https:" + "//download.pytorch.org/whl/cu124"
PYPI_INDEX = "https:" + "//pypi.org/simple"
torch_output = install(
    "PyTorch 2.5.1 CUDA 12.4",
    "--index-url", PYTORCH_INDEX,
    "--extra-index-url", PYPI_INDEX,
    "--requirement",
    "/content/LLM_Router/ml/requirements-training-cu124.txt",
)
library_output = install(
    "pinned evaluation libraries",
    "--requirement",
    "/content/LLM_Router/ml/requirements-training.lock",
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "uninstall", "--yes",
        "--break-system-packages", "torchvision", "torchaudio",
    ],
    check=False,
)
install(
    "LLM Router package",
    "--no-deps", "-e", "/content/LLM_Router/ml",
)

verify_code = """
import json
import torch
import transformers
print(json.dumps({
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'transformers': transformers.__version__,
    'cuda_available': torch.cuda.is_available(),
}, sort_keys=True))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
"""
verification = subprocess.run(
    [sys.executable, "-c", verify_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print("[fresh-process verification]", verification.stdout, flush=True)
if verification.returncode != 0:
    raise RuntimeError("Fresh-process runtime verification failed")

compiled_packages_changed = any(
    marker in (torch_output + library_output)
    for marker in ("Successfully installed", "Attempting uninstall")
)
restart_marker = Path("/content/.llm_router_epoch1_eval_restart_required")
if compiled_packages_changed:
    restart_marker.write_text(str(os.getpid()), encoding="utf-8")
    print(
        "IMPORTANT: 패키지가 교체되었습니다. Colab kernel을 재시작한 뒤 Run All 하세요.",
        flush=True,
    )
else:
    print("Runtime packages are ready.", flush=True)

In [ ]:
import os
from pathlib import Path

restart_marker = Path("/content/.llm_router_epoch1_eval_restart_required")
if restart_marker.is_file():
    installed_in_pid = restart_marker.read_text(encoding="utf-8").strip()
    if installed_in_pid == str(os.getpid()):
        raise RuntimeError(
            "설치 직후 kernel입니다. Colab kernel을 재시작한 뒤 Run All 하세요."
        )
    restart_marker.unlink()

TRAINING_PATH = Path(
    "/content/LLM_Router/ml/artifacts/routerbench-training/"
    "withmartian-routerbench-2024-03-27__minimum-sufficient-tier-v1__"
    "routerbench-training-preprocess-v1/training.parquet"
)
ROUTERARENA_PATH = Path(
    "/content/LLM_Router/ml/artifacts/routerarena-evaluation-full/evaluation.parquet"
)
ROUTERARENA_METADATA_PATH = Path(
    "/content/LLM_Router/ml/artifacts/routerarena-evaluation-full/metadata.parquet"
)
ROUTERARENA_CONFIG_PATH = Path(
    "/content/LLM_Router/ml/config/routerarena_evaluation.json"
)
EPOCH1_CHECKPOINT = DRIVE_RUN / "checkpoints" / "step-00003476"

required_paths = [
    TRAINING_PATH,
    ROUTERARENA_PATH,
    ROUTERARENA_METADATA_PATH,
    ROUTERARENA_CONFIG_PATH,
    EPOCH1_CHECKPOINT,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
if not (EPOCH1_CHECKPOINT / "_SUCCESS").is_file():
    raise RuntimeError(
        f"완료되지 않은 epoch 1 checkpoint입니다: {EPOCH1_CHECKPOINT}"
    )

print({
    "epoch1_checkpoint": str(EPOCH1_CHECKPOINT),
    "checkpoint_complete": True,
    "routerarena_batch_size": 1,
    "routerarena_max_length": 8192,
})

In [ ]:
from llm_router_labeling.evaluation.routerarena_epoch_evaluation import (
    RouterArenaEpochEvaluator,
)

LOCAL_EVALUATION_ROOT = Path(
    "/content/llm-router-training/modernbert-router-baseline-v1/"
    "routerarena-epoch-evaluations"
)
DRIVE_EVALUATION_ROOT = DRIVE_RUN / "routerarena-epoch-evaluations"

evaluator = RouterArenaEpochEvaluator(
    training_path=TRAINING_PATH,
    evaluation_path=ROUTERARENA_PATH,
    metadata_path=ROUTERARENA_METADATA_PATH,
    config_path=ROUTERARENA_CONFIG_PATH,
    local_output_dir=LOCAL_EVALUATION_ROOT,
    persistent_output_dir=DRIVE_EVALUATION_ROOT,
    batch_size=1,
    max_length=8192,
    device="cuda",
)

print("Starting epoch 1 RouterArena evaluation...", flush=True)
evaluated_now = evaluator.recover_epoch_from_checkpoint(EPOCH1_CHECKPOINT)
print({
    "evaluated_now": evaluated_now,
    "artifact_dir": str(DRIVE_EVALUATION_ROOT / "epoch-01"),
})

In [ ]:
import json
import pandas as pd
from IPython.display import SVG, display

EPOCH1_RESULT = DRIVE_EVALUATION_ROOT / "epoch-01"
METRICS_PATH = EPOCH1_RESULT / "metrics.json"
if not METRICS_PATH.is_file():
    raise FileNotFoundError(f"평가 metric이 없습니다: {METRICS_PATH}")
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))

summary = {
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "macro_precision": metrics["macro_precision"],
    "macro_recall": metrics["macro_recall"],
    "under_routing_rate": metrics["under_routing_rate"],
    "over_routing_rate": metrics["over_routing_rate"],
    "mean_absolute_tier_error": metrics["mean_absolute_tier_error"],
    "predicted_label_distribution": metrics["predicted_label_distribution"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

per_class = pd.DataFrame(metrics["per_class"]).T
per_class.index.name = "label"
display(per_class)

confusion = pd.DataFrame(metrics["confusion_matrix"]).T
confusion.index.name = "actual"
confusion.columns.name = "predicted"
display(confusion)

display(SVG(filename=str(EPOCH1_RESULT / "predicted_group_distribution.svg")))
display(SVG(filename=str(EPOCH1_RESULT / "predicted_group_heatmap.svg")))

## 이후 학습 재개

전체 학습 notebook의 `--routerarena-batch-size`를 반드시 `1`로 유지하고 같은 `DRIVE_RUN`에서 `--resume-mode auto`로 실행합니다. 이 notebook이 만든 `epoch-01/_SUCCESS` artifact를 감지하면 epoch 1 RouterArena 평가는 건너뛰고 epoch 2 학습부터 이어갑니다.